# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object
meta = dataset.metadata

print(f"Dataset title: {meta.name}\n")
print(f"Dataset description: {meta.description}\n")
print(f"Dataset DOI: {getattr(meta, 'identifier', 'Not specified')}")
print(f"Published: {getattr(meta, 'datePublished', 'Not specified')}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields (columns) each contains.

In [ ]:
# Get all record sets and their IDs
record_sets = dataset.record_sets
print("Available record sets and fields:")

for recset in record_sets:
    print(f"- Record set name: {recset.name}  |  @id: {recset.id}")
    print("  Fields (columns):")
    for field in recset.fields:
        print(f"    - {field.name}  |  @id: {field.id}")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Each record set and field is identified by its Croissant schema `@id`.

In [ ]:
# Let's list all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids:")
print(record_set_ids)

# For this dataset, we commonly have one main table. We'll extract all for demonstration:
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set with @id: {rs_id}\nColumns: {df.columns.tolist()}\n")
    else:
        print(f"Record set {rs_id} has no records.\n")

# For continued exploration, let's choose the primary record set if present
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Main record set selected: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Let us conduct some basic filtering and normalization on numeric fields. This will involve:
- Listing all numeric columns (fields of type `Integer` or `Float` as specified in the Croissant schema).
- Filtering for values greater than a threshold.
- Normalizing a selected numeric field.
- Optionally grouping by a categorical field.

All field and group IDs are referenced by their `@id`.

In [ ]:
# Identify numeric fields (Float or Integer types) for the selected main record set
numeric_field_ids = []
categorical_field_ids = []

for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            dtype = getattr(field, 'data_type', None)
            if dtype in ('schema:Float', 'schema:Integer', 'Float', 'Integer'):
                numeric_field_ids.append(field.id)
            else:
                categorical_field_ids.append(field.id)
        break

print(f"Numeric fields (@id): {numeric_field_ids}")
print(f"Categorical fields (@id): {categorical_field_ids}\n")

# We'll proceed if we have at least one numeric field
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    print(f"Using numeric field for EDA: {numeric_field}")
    df = dataframes[main_record_set_id]
    
    # Some columns may still have str/object data, try conversion
    df = df.copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].quantile(0.75)  # Use 3rd quartile as an example threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (75th percentile): {len(filtered_df)} records \n")
    print(filtered_df[[numeric_field]].head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records (first 5):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    if categorical_field_ids:
        group_field = categorical_field_ids[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values. We'll also show group-wise means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_ids:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of field '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Histogram of the normalized field (filtered set)
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True)
        plt.title(f"Normalized distribution of '{numeric_field}' (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.show()

    # Grouped barplot if grouped_df is available
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        grouped_df.plot(kind='bar')
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded a FAIR² dataset defined with the Croissant schema using the `mlcroissant` library
- Explored the available record sets and fields, referencing all with their `@id`
- Extracted the main table into a DataFrame, performed filtering and normalization on a numeric field
- Visualized data distributions and provided a basic group-wise summary

For other data processing and advanced analyses, refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) and review the Croissant schema to see all available metadata and process instructions.